# Day 17: GPU Architecture — SMs, Memory Hierarchy, HBM
> *Inference Engineering* — Chapter 3.1 | Philip Kiely, Baseten Books 2026

**Layer:** Infrastructure | **Prerequisite:** Day 04 (Arithmetic Intensity)


## Concept Overview

A modern GPU (Hopper H100) has 132 Streaming Multiprocessors (SMs), each with 128 CUDA cores. The memory hierarchy spans registers (fastest, per-thread) → L1/shared SRAM (per-SM) → L2 cache → HBM (high-bandwidth memory). HBM is the AI accelerator's defining feature: 3.35 TB/s bandwidth on H100 vs ~100 GB/s for CPU DRAM. Understanding this hierarchy is key to writing efficient kernels and understanding why roofline analysis matters.


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import time

print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'SM count: {props.multi_processor_count}')
    print(f'Total memory: {props.total_memory/1e9:.1f} GB')
    print(f'Max threads per SM: {props.max_threads_per_multi_processor}')
    print(f'Shared memory per SM: {props.shared_memory_per_multiprocessor/1024:.0f} KB')
    print(f'L2 cache: {props.l2_cache_size/1e6:.0f} MB')


## 1. GPU Memory Hierarchy

Access latency and bandwidth at each level of the hierarchy determines what can be computed efficiently.


In [ ]:
# GPU Memory Hierarchy (H100 SXM)
hierarchy = [
    ('Registers',       '256 KB/SM',  '~1 cycle',    'N/A',          'per-thread'),
    ('L1/Shared SRAM',  '228 KB/SM',  '~30 cycles',  '33 TB/s',      'per-SM'),
    ('L2 Cache',        '50 MB',      '~100 cycles', '12 TB/s',      'all SMs'),
    ('HBM3',            '80 GB',      '~400 cycles', '3.35 TB/s',    'all SMs'),
    ('NVLink',          'N/A',        '~1000 cycles','900 GB/s',     'multi-GPU'),
    ('PCIe 5.0',        'N/A',        '~100K cycles','128 GB/s',     'CPU-GPU'),
    ('CPU DRAM',        '768 GB',     '~300 cycles', '~100 GB/s',    'host'),
]

print('H100 SXM Memory Hierarchy:')
print(f'{"Level":<22} {"Capacity":<15} {"Latency":<15} {"Bandwidth":<15} {"Scope":<15}')
print('-' * 82)
for level, cap, lat, bw, scope in hierarchy:
    print(f'{level:<22} {cap:<15} {lat:<15} {bw:<15} {scope:<15}')


## 2. Streaming Multiprocessor (SM) Anatomy

Each SM has CUDA cores for FP32/INT32, Tensor Cores for matrix ops, special function units (SFUs), and shared SRAM.


In [ ]:
# SM utilization model
def sm_occupancy(threads_per_block, registers_per_thread, shared_mem_per_block,
                max_threads_per_sm=2048, max_regs_per_sm=65536, max_shared_mem_per_sm=228*1024):
    """Estimate SM occupancy (fraction of max concurrent warps)."""
    warps_per_block = (threads_per_block + 31) // 32
    # Limits from different resources
    limit_threads = max_threads_per_sm // threads_per_block
    limit_regs = max_regs_per_sm // (registers_per_thread * threads_per_block) if registers_per_thread > 0 else 999
    limit_shared = max_shared_mem_per_sm // shared_mem_per_block if shared_mem_per_block > 0 else 999
    max_blocks = min(limit_threads, limit_regs, limit_shared, 32)  # max 32 blocks per SM
    active_warps = max_blocks * warps_per_block
    max_warps = max_threads_per_sm // 32
    occupancy = min(active_warps, max_warps) / max_warps
    return occupancy, max_blocks, min(limit_threads, limit_regs, limit_shared)

print('SM Occupancy Analysis:')
print(f'{"Threads/Block":>15} {"Regs/Thread":>13} {"Shared (KB)":>13} {"Occupancy":>11} {"Limiter":>12}')
for (tpb, rpt, smem_kb) in [
    (128,  32, 4),
    (256,  64, 4),
    (512,  32, 0),
    (1024, 32, 0),
    (256,  64, 64),
    (128, 128, 0),
]:
    occ, max_b, limiter_count = sm_occupancy(tpb, rpt, smem_kb*1024)
    print(f'{tpb:>15} {rpt:>13} {smem_kb:>13} {occ:>11.0%} {limiter_count:>12}')


## 3. Measuring HBM Bandwidth in Practice

Measure how close to peak HBM bandwidth a memory-bound operation achieves on the target GPU.


In [ ]:
def measure_hbm_bandwidth(size_mb=512):
    if not torch.cuda.is_available():
        print('GPU not available')
        return

    n = size_mb * 1024 * 1024 // 2
    a = torch.randn(n, dtype=torch.float16, device='cuda')
    b = torch.empty_like(a)

    # Warmup
    for _ in range(5):
        b.copy_(a)
    torch.cuda.synchronize()

    times = []
    for _ in range(20):
        t0 = time.perf_counter()
        b.copy_(a)
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)

    bytes_moved = 2 * n * 2  # read + write
    bw_gb_s = bytes_moved / np.median(times) / 1e9
    print(f'HBM Bandwidth measurement ({size_mb} MB transfer):')
    print(f'  Median time: {np.median(times)*1000:.3f} ms')
    print(f'  Bandwidth: {bw_gb_s:.1f} GB/s')
    print(f'  (DGX Spark spec: ~273 GB/s)')
    print(f'  Efficiency: {bw_gb_s/273*100:.0f}% of peak')

measure_hbm_bandwidth()


## Experiments: Try These

1. **L2 cache effects**: Measure bandwidth for tensor sizes from 1KB to 1GB. Plot bandwidth vs size — identify where L2 cache and HBM transitions occur.
2. **Occupancy optimization**: Profile a simple kernel with `torch.profiler`. Look at `sm_efficiency` and `achieved_occupancy` metrics.
3. **Warp divergence**: Write two versions of a conditional kernel — one with aligned conditions (all warps take same branch) and one with divergence. Measure the throughput difference.


## Key Takeaways

- GPUs have 100s of SMs, each with 1000s of CUDA cores — parallelism happens at the warp level (32 threads executing in lockstep).
- The memory hierarchy spans registers → shared SRAM → L2 → HBM, with 3-400x bandwidth difference between levels.
- HBM (High Bandwidth Memory) stacked on-package is what makes GPUs viable for LLM inference: 3.35 TB/s on H100 vs 100 GB/s for CPU DRAM.
- SM occupancy determines how well the GPU can hide memory latency — high occupancy is needed for memory-bound kernels.

## References
- *Inference Engineering* Ch 3.1 — Philip Kiely, Baseten Books 2026
- NVIDIA H100 Architecture White Paper
- CUDA Programming Guide: Memory Hierarchy
